In [1]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
os.chdir('..')

/raid/home/m13521157/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [3]:
from typing import List, Dict, Any, Literal

In [4]:
def parse_absa_string(text: str):
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

def convert_to_gas_format(data_list, spaced=False):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A, O, S); (A, O, S); ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	if spaced:
		triplets = [f"( {item['A']} | {item['O']} | {item['S']} )" for item in data_list]
	else:
		triplets = [f"({item['A']}| {item['O']}| {item['S']})" for item in data_list]

	# Join the list of strings together with a semicolon and space
	if spaced:
		return " ; ".join(triplets)
	return "; ".join(triplets)

In [21]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw'  # Replace with your actual Google Sheet ID
gid_indo_train = '1757632097'  # Replace with the actual GID for the Indonesian sheet
gid_indo_test = '440710234'  # Replace with the actual GID for the Indonesian sheet
lang = 'indo'
split = 'train'
corrected_data = []
try:
	df_corrected = get_google_sheet(google_sheet_id, globals()[f'gid_{lang}_{split}'])
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

## Construct MvP Format from corrected sheets

In [22]:
df_corrected

,sentence_id,element_order,input,targets,corrected_targets,terms_exist,Unnamed: 6,Unnamed: 7
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] tidak berfungsi optimal [S] negativ...,True,NaN,NaN
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,True,NaN,NaN
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,True,NaN,NaN
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] semuanya [O] nyaman [S] positive\n[A] null...,True,NaN,NaN
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,[A] snack [O] tidak dapat [S] negative,True,NaN,NaN
...,...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,[A] wifi [O] kurang joss [S] negative,True,NaN,NaN
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,[A] kamar [O] cukup bersih [S] positive\n[A] k...,True,NaN,NaN
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,[A] pelayananya [O] sangat ramah [S] positive\...,True,NaN,NaN
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,True,NaN,NaN


In [23]:
new_dataset = []
for idx, row in df_corrected.iterrows():
    new_target = row['corrected_targets']
    splitted_target = new_target.split('\n')
    splitted_target = [target.strip() for target in splitted_target]
    splitted_target = [target for target in splitted_target if target]
    new_dataset.append({
        'sentence_id': row['sentence_id'],
        'instance_id': row['sentence_id'] * 5,
        'input': row['input'],
        'target': ' [SSEP] '.join(splitted_target),
        'element_order': row['element_order'],
        'task_elements': 'aos'
	})

In [24]:
# Write the new_dataset to a json file
dataset_path = f'dataset/hotel_reviews/indo/mvp_aos/{split}.json'
with open(dataset_path, 'w') as f:
    json.dump(new_dataset, f, indent=4, ensure_ascii=False)
print(f"Converted dataset saved to {dataset_path}")

Converted dataset saved to dataset/hotel_reviews/indo/mvp_aos/train.json


## Construct from MvP format

### GAS

In [53]:
lang = 'indo'
dataset_type = 'hotel_reviews_aug'
dataset_folder = 'mvp_aos'
dataset_per_split = {}
splits = ['train', 'dev', 'test']

In [54]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'dataset/{dataset_type}/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['dataset/hotel_reviews_aug/indo/mvp_aos/train.json']
Reading files for dev : ['dataset/hotel_reviews_aug/indo/mvp_aos/dev.json']
Reading files for test : ['dataset/hotel_reviews_aug/indo/mvp_aos/test.json']


In [55]:
def convert_to_gas_format(data_list):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A | O | S) ; (A | O | S) ; ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	triplets = [f"( {item['A']} | {item['O']} | {item['S']} )" for item in data_list]
	
	# Join the list of strings together with a semicolon and space
	return " ; ".join(triplets)

In [56]:
gas_data_per_split = {}
for split in splits:
	gas_data_per_split[split] = []
	instance_id_counter = dataset_per_split[split][0]['sentence_id']
	for instance in dataset_per_split[split]:
		absa_string = instance['target']
		parsed_absa = parse_absa_string(absa_string)
		try:
			for d in parsed_absa:
				assert all(key in d.keys() for key in ['A', 'O', 'S'])
		except:
			print("Error in instance:", instance)
			raise ValueError("Parsed ABSA contains invalid keys.")
		gas_format = convert_to_gas_format(parsed_absa)
		gas_data_per_split[split].append({
			"sentence_id": instance['sentence_id'],
			"instance_id": instance_id_counter,
			'task_elements': instance['task_elements'],
			"input": f"{instance['input'].replace('[A] [O] [S]', '').strip()}", # Add arrow for input target delimiter (remove if not needed)
			"target": gas_format.strip(),
			"element_order": instance['element_order']
		})
		instance_id_counter += 1

In [57]:
len(gas_data_per_split['train']), len(gas_data_per_split['dev']), len(gas_data_per_split['test'])

(16826, 1000, 1000)

In [58]:
for split in splits:
    gas_dataset_path = f'dataset/{dataset_type}/{lang}/gas/{split}.json'
    os.makedirs(os.path.dirname(gas_dataset_path), exist_ok=True)
    with open(gas_dataset_path, 'w') as f:
        json.dump(gas_data_per_split[split], f, ensure_ascii=False, indent=4)
    print(f"GAS formatted dataset saved to {gas_dataset_path}")

GAS formatted dataset saved to dataset/hotel_reviews_aug/indo/gas/train.json
GAS formatted dataset saved to dataset/hotel_reviews_aug/indo/gas/dev.json
GAS formatted dataset saved to dataset/hotel_reviews_aug/indo/gas/test.json


### Lego-ABSA

In [59]:
lang = 'indo'
dataset_type = 'hotel_reviews_aug'
dataset_folder = 'mvp_aos'
dataset_per_split = {}
splits = ['train', 'dev', 'test']

In [60]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'dataset/{dataset_type}/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['dataset/hotel_reviews_aug/indo/mvp_aos/train.json']
Reading files for dev : ['dataset/hotel_reviews_aug/indo/mvp_aos/dev.json']
Reading files for test : ['dataset/hotel_reviews_aug/indo/mvp_aos/test.json']


In [61]:

element_order_dict = {
    'indolegoabsa_multitask': ['aos', 'ao', 'as', 'a', 's'],
    'legoabsa_multitask': ['aos', 'ao', 'as'],
    'legoabsa_tasktransfer': ['oa', 'as']

}

In [62]:
special_tokens = {
	'a': '<|aspect|>',
	'o': '<|opinion|>',
	's': '<|sentiment|>'
}

initial_definitions = {
    'a': 'aspect',
    'o': 'opinion',
    's': 'sentiment'
}

In [63]:
def convert_output_to_legoabsa_format(data_list, order='aos'):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the Lego-ABSA paper.

	Args:
		data_list: A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order: A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""
	global special_tokens
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	# Build triplets based on the order length
	triplets = []
	for item in data_list:
		parts = []
		for element in tuple_order:
			if element.upper() in item:
				parts.append(f"{special_tokens[element]} {item[element.upper()].strip()}")
		triplets.append(" ".join(parts))
	
	# Join the list of strings together with a semicolon and space
	return ";".join(triplets)

def convert_input_to_legoabsa_format(input_str, order='aos'):
	global special_tokens
	global initial_definitions
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	input_str = input_str.replace('[A] [O] [S]', '').strip()
	
	# Make replacements based on the order like this '{input_str}|aspect: <|box_start|> , opinion: <|quad_start|> , sentiment: <|vision_start|>'
	definitions = ' , '.join([f"{initial_definitions[element]}: {special_tokens[element]}" for element in tuple_order])
	return f"{input_str}| {definitions}"

In [64]:
convert_input_to_legoabsa_format("The room was clean but the service was terrible. The location is great though.", order='sa')

'The room was clean but the service was terrible. The location is great though.| sentiment: <|sentiment|> , aspect: <|aspect|>'

In [65]:
convert_output_to_legoabsa_format([{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
 {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}], order='aos')

'<|aspect|> harga <|opinion|> terjangkau <|sentiment|> positive;<|aspect|> fasilitas <|opinion|> nyaman <|sentiment|> positive'

In [66]:
for element_order_key, element_orders in element_order_dict.items():
	lego_absa_data_per_split = {}

	# Initialize train split data
	lego_absa_data_per_split['train'] = []
	instance_id_counter = dataset_per_split['train'][0]['sentence_id']
	for element_order in element_orders:
		for instance in dataset_per_split['train']:
			absa_string = instance['target']
			parsed_absa = parse_absa_string(absa_string)
			try:
				for d in parsed_absa:
					assert all(key in d.keys() for key in ['A', 'O', 'S'])
			except:
				print("Error in instance:", instance)
				raise ValueError("Parsed ABSA contains invalid keys.")
			lego_absa_format = convert_output_to_legoabsa_format(parsed_absa, order=element_order)
			input_legoabsa_format = convert_input_to_legoabsa_format(instance['input'], order=element_order)
			lego_absa_data_per_split['train'].append({
				"sentence_id": instance['sentence_id'],
				"instance_id": instance_id_counter,
				'task_elements': instance['task_elements'],
				"input": input_legoabsa_format.strip() + ' =>',
				"target": lego_absa_format.strip(),
				"element_order": element_order
			})
			instance_id_counter += 1
	
	# Make dev and test data
	for split in ['dev', 'test']:
		lego_absa_data_per_split[split] = []
		instance_id_counter = len(lego_absa_data_per_split['train']) + 0 if split == 'dev' else len(lego_absa_data_per_split['dev']) # Start test IDs after train IDs + dev IDs (dev have 1000 instances)
		for instance in dataset_per_split[split]:
			absa_string = instance['target']
			parsed_absa = parse_absa_string(absa_string)
			try:
				for d in parsed_absa:
					assert all(key in d.keys() for key in ['A', 'O', 'S'])
			except:
				print("Error in instance:", instance)
				raise ValueError("Parsed ABSA contains invalid keys.")
			lego_absa_format = convert_output_to_legoabsa_format(parsed_absa, order='aos')
			input_legoabsa_format = convert_input_to_legoabsa_format(instance['input'], order='aos')
			lego_absa_data_per_split[split].append({
				"sentence_id": instance['sentence_id'],
				"instance_id": instance_id_counter,
				'task_elements': instance['task_elements'],
				"input": input_legoabsa_format.strip(),
				"target": lego_absa_format.strip(),
				"element_order": 'aos'
			})
			instance_id_counter += 1
	
	# Check length
	print(lego_absa_data_per_split['train'].__len__(), lego_absa_data_per_split['dev'].__len__(), lego_absa_data_per_split['test'].__len__())

	for split in splits:
		legoabsa_dataset_path = f'dataset/{dataset_type}/{lang}/{element_order_key}/{split}.json'
		os.makedirs(os.path.dirname(legoabsa_dataset_path), exist_ok=True)
		with open(legoabsa_dataset_path, 'w') as f:
			json.dump(lego_absa_data_per_split[split], f, ensure_ascii=False, indent=4)
		print(f"Lego-ABSA formatted dataset saved to {legoabsa_dataset_path}")

84130 1000 1000
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/indolegoabsa_multitask/train.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/indolegoabsa_multitask/dev.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/indolegoabsa_multitask/test.json
50478 1000 1000
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_multitask/train.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_multitask/dev.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_multitask/test.json
33652 1000 1000
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_tasktransfer/train.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_tasktransfer/dev.json
Lego-ABSA formatted dataset saved to dataset/hotel_reviews_aug/indo/legoabsa_tasktransfer/test.json
